# BEAM Linkstats Consist Analysis

This notebook bootstraps a Consist tracker against an **archived PILATES run** (scratch storage), lists linkstats artifacts by facets, and computes first-pass summary + delta metrics over phys-sim iterations.

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)
sns.set_theme(style="whitegrid")

print("Kernel Python:", sys.executable)


In [ ]:
# --- Configure these for your environment/run ---
USER = os.environ.get("USER", "$USER")
RUN_NAME = "new-asim-consist-flash-20260209-103617"
SLURM_JOB_ID = os.environ.get("SLURM_JOB_ID", "")

OUTPUT_ROOT = Path(f"/global/scratch/users/{USER}/pilates-output").resolve()
PROJECT_ROOT = Path(f"/global/scratch/users/{USER}/sources/PILATES").resolve()
ARCHIVE_RUN_DIR = OUTPUT_ROOT / RUN_NAME

# DB path resolution order:
# 1) PILATES_DB_PATH env override
# 2) node-local default (/local/job$SLURM_JOB_ID/db/pilates-analysis.duckdb)
# 3) archived run DB
DB_PATH_ENV = os.environ.get("PILATES_DB_PATH")
LOCAL_DB_PATH = Path(f"/local/job{SLURM_JOB_ID}/db/pilates-analysis.duckdb") if SLURM_JOB_ID else None
ARCHIVE_DB_PATH = ARCHIVE_RUN_DIR / "provenance.duckdb"

if DB_PATH_ENV:
    DB_PATH = Path(DB_PATH_ENV).expanduser()
elif LOCAL_DB_PATH is not None and LOCAL_DB_PATH.exists():
    DB_PATH = LOCAL_DB_PATH
else:
    DB_PATH = ARCHIVE_DB_PATH

TARGET_YEAR = 2018

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

print("CWD:", Path.cwd())
print("PROJECT_ROOT:", PROJECT_ROOT)
print("ARCHIVE_RUN_DIR:", ARCHIVE_RUN_DIR)
print("LOCAL_DB_PATH:", LOCAL_DB_PATH)
print("DB_PATH:", DB_PATH)


In [ ]:
from pilates.utils.consist_analysis import (
    assign_effective_beam_sub_iteration,
    build_linkstats_hourly_pca_matrix,
    create_analysis_tracker,
    find_linkstats_artifacts,
    print_duckdb_health,
    summarize_linkstats_artifacts,
    summarize_linkstats_traveltime_deltas,
    summarize_linkstats_traveltime_deltas_hourly_weighted,
)

db_health = print_duckdb_health(db_path=DB_PATH, probe_open=True)

tracker = create_analysis_tracker(
    db_path=DB_PATH,
    archive_run_dir=ARCHIVE_RUN_DIR,
    project_root=PROJECT_ROOT,
    output_root=OUTPUT_ROOT,
)

print("Tracker mounts:")
for name, root in tracker.mounts.items():
    print(f"  {name}:// -> {root}")


In [ ]:
# Primary query: phys-sim unmodified linkstats parquet artifacts for one year across all outer iterations
artifacts_df = find_linkstats_artifacts(
    tracker,
    year=TARGET_YEAR,
    artifact_family="linkstats_unmodified_phys_sim_iter_parquet",
    namespace="beam",
    limit=20000,
)

print(f"Found {len(artifacts_df)} artifacts")
artifacts_df.head(20)

In [ ]:
# Quick facet/view sanity check
if artifacts_df.empty:
    print("No linkstats artifacts found with the requested filters.")
else:
    display(
        artifacts_df[[
            "key",
            "container_uri",
            "facet_source",
            "phys_sim_iteration",
            "beam_sub_iteration",
        ]].head(25)
    )
    print("Facet source summary:")
    display(artifacts_df["facet_source"].value_counts(dropna=False))


In [ ]:
# Keep only indexed snapshots and derive an effective sub-iteration index.
# Promoted final sub-iteration rows (beam_sub_iteration = NaN) are mapped to
# max(non-null sub-iteration) + 1 within each (run_id, year, iteration, phys_sim_iteration).
artifacts_clean = artifacts_df[artifacts_df["facet_source"] == "artifact_kv"].copy()
artifacts_clean = assign_effective_beam_sub_iteration(artifacts_clean)

# Keep raw value for debugging, but use effective value for analysis grouping.
artifacts_clean["beam_sub_iteration_raw"] = artifacts_clean["beam_sub_iteration"]
artifacts_clean["beam_sub_iteration"] = artifacts_clean["beam_sub_iteration_effective"]

# Optional logical de-dup in case multiple keys map to the same snapshot.
artifacts_clean = (
    artifacts_clean
    .sort_values(["run_id", "year", "iteration", "phys_sim_iteration", "beam_sub_iteration", "key"])
    .drop_duplicates(
        subset=["run_id", "year", "iteration", "phys_sim_iteration", "beam_sub_iteration"],
        keep="first",
    )
    .reset_index(drop=True)
)

print(f"Raw artifacts: {len(artifacts_df)}")
print(f"Filtered artifacts: {len(artifacts_clean)}")

# Use volume-weighted travel time in per-artifact summary stats.
summary_df = summarize_linkstats_artifacts(
    artifacts_clean,
    tracker=tracker,
    traveltime_weighting="volume_weighted",
)
print(f"Computed summaries for {len(summary_df)} artifacts via Consist views")
summary_df.head(20)


In [ ]:
# First-pass rollup by sub-iteration + phys-sim iteration
if summary_df.empty:
    print("No summary rows available. Check mounts/path resolution above.")
else:
    rollup = (
        summary_df.groupby(["year", "iteration", "beam_sub_iteration", "phys_sim_iteration"], dropna=False)
        .agg(
            artifacts=("key", "count"),
            rows_total=("row_count", "sum"),
            distinct_links_mean=("distinct_links", "mean"),
            volume_sum_total=("volume_sum", "sum"),
            vmt_miles_total=("vmt_miles", "sum"),
            vht_hours_total=("vht_hours", "sum"),
            total_delay_hours_total=("total_delay_hours", "sum"),
            vmt_per_vht_mph_mean=("vmt_per_vht_mph", "mean"),
            vht_per_vmt_hours_per_mile_mean=("vht_per_vmt_hours_per_mile", "mean"),
            traveltime_mean_mean=("traveltime_mean", "mean"),
            traveltime_p95_mean=("traveltime_p95", "mean"),
        )
        .assign(
            vmt_per_vht_mph_total=lambda df: df["vmt_miles_total"] / df["vht_hours_total"],
            vht_per_vmt_hours_per_mile_total=lambda df: df["vht_hours_total"] / df["vmt_miles_total"],
        )
        .reset_index()
        .sort_values(["year", "iteration", "beam_sub_iteration", "phys_sim_iteration"], na_position="last")
    )
    display(rollup)


In [ ]:
# Build PCA matrix: rows=artifacts, cols=(selected links x hourly bins)
PCA_LINK_FILTER_STRATEGY = "all"  # "first" | "last" | "all"
PCA_VOLUME_COVERAGE = 0.98
PCA_HOUR_GROUP_SIZE = 1  # 1 => 24 bins
PCA_METRIC = "speed_mph"

pca_payload = build_linkstats_hourly_pca_matrix(
    summary_df,
    tracker=tracker,
    link_filter_strategy=PCA_LINK_FILTER_STRATEGY,
    volume_coverage=PCA_VOLUME_COVERAGE,
    hour_group_size=PCA_HOUR_GROUP_SIZE,
    metric=PCA_METRIC,
    dtype="float32",
)

X = pca_payload["matrix"]
artifact_index_df = pca_payload["artifact_index"]
feature_index_df = pca_payload["feature_index"]

print("PCA matrix shape:", X.shape)
print("Artifacts:", len(artifact_index_df))
print("Selected links:", len(pca_payload["link_index"]))
print("Hour bins:", pca_payload["hour_bins"])

display(artifact_index_df.head(10))
display(feature_index_df.head(10))


In [ ]:
# Column-volume-weighted PCA using NumPy SVD
if X.size == 0 or X.shape[0] < 2 or X.shape[1] == 0:
    print("Not enough data to run PCA.")
else:
    X64 = X.astype(np.float64, copy=False)
    X_centered = X64 - X64.mean(axis=0, keepdims=True)

    w = pca_payload["column_weights_normalized"].astype(np.float64, copy=False)
    if w.size != X_centered.shape[1]:
        raise ValueError("Column-weight length does not match feature count")

    sqrt_w = np.sqrt(np.clip(w, 0.0, None))
    X_weighted = X_centered * sqrt_w[None, :]

    U, S, Vt = np.linalg.svd(X_weighted, full_matrices=False)
    denom = max(X_weighted.shape[0] - 1, 1)
    explained_var = (S ** 2) / denom
    total_var = explained_var.sum()
    explained_ratio = explained_var / total_var if total_var > 0 else np.zeros_like(explained_var)

    n_components = min(5, U.shape[1])
    scores = U[:, :n_components] * S[:n_components]

    pca_scores_df = artifact_index_df.copy()
    for i in range(n_components):
        pca_scores_df[f"PC{i + 1}"] = scores[:, i]

    explained_df = pd.DataFrame({
        "component": [f"PC{i + 1}" for i in range(n_components)],
        "explained_variance": explained_var[:n_components],
        "explained_variance_ratio": explained_ratio[:n_components],
    })

    display(explained_df)
    display(pca_scores_df.head(20))

    if n_components >= 2:
        plt.figure(figsize=(8, 6))
        sns.scatterplot(
            data=pca_scores_df,
            x="PC1",
            y="PC2",
            hue="iteration",
            style="beam_sub_iteration",
            s=80,
        )
        plt.title("Weighted PCA Scores (PC1 vs PC2)")
        plt.tight_layout()


In [ ]:
# PCA exploratory diagnostics and loadings
if not all(name in globals() for name in ["X", "Vt", "pca_scores_df", "explained_df", "feature_index_df"]):
    print("PCA outputs missing. Run the PCA cell first.")
elif X.size == 0:
    print("PCA matrix is empty. Skipping diagnostics.")
else:
    explained_plot_df = explained_df.copy()
    explained_plot_df["cumulative_explained_variance_ratio"] = explained_plot_df["explained_variance_ratio"].cumsum()
    display(explained_plot_df)

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    sns.barplot(data=explained_plot_df, x="component", y="explained_variance_ratio", ax=axes[0], color="#4C72B0")
    axes[0].set_title("Explained Variance Ratio")
    axes[0].set_ylabel("ratio")
    axes[0].set_xlabel("component")

    sns.lineplot(data=explained_plot_df, x="component", y="cumulative_explained_variance_ratio", marker="o", ax=axes[1], color="#55A868")
    axes[1].set_title("Cumulative Explained Variance")
    axes[1].set_ylim(0, 1.01)
    axes[1].set_ylabel("cumulative ratio")
    axes[1].set_xlabel("component")
    plt.tight_layout()

    n_plot_components = min(3, Vt.shape[0])
    loadings_df = pd.DataFrame({"feature_index": np.arange(Vt.shape[1], dtype=int)})
    for i in range(n_plot_components):
        loadings_df[f"PC{i + 1}_loading"] = Vt[i, :]
        loadings_df[f"PC{i + 1}_abs_loading"] = np.abs(Vt[i, :])

    pca_feature_loadings_df = feature_index_df.merge(loadings_df, on="feature_index", how="left")
    pca_link_importance_df = (
        pca_feature_loadings_df
        .groupby("link", dropna=False)
        .agg(
            daily_volume=("daily_volume", "first"),
            cumulative_share=("cumulative_share", "first"),
            PC1_abs_loading_mean=("PC1_abs_loading", "mean"),
            PC1_abs_loading_max=("PC1_abs_loading", "max"),
            PC2_abs_loading_mean=("PC2_abs_loading", "mean") if "PC2_abs_loading" in pca_feature_loadings_df.columns else ("PC1_abs_loading", "mean"),
        )
        .reset_index()
        .sort_values("PC1_abs_loading_mean", ascending=False)
    )

    print("Top links by mean absolute PC1 loading:")
    display(pca_link_importance_df.head(20))

    hourly_profile = (
        pca_feature_loadings_df
        .groupby("hour_bin", dropna=False)
        .agg(PC1_abs_loading_mean=("PC1_abs_loading", "mean"))
        .reset_index()
        .sort_values("hour_bin")
    )
    plt.figure(figsize=(8, 4))
    sns.lineplot(data=hourly_profile, x="hour_bin", y="PC1_abs_loading_mean", marker="o")
    plt.title("Mean Absolute PC1 Loading by Hour Bin")
    plt.xlabel("hour_bin")
    plt.ylabel("mean |loading|")
    plt.tight_layout()

    if pca_scores_df.shape[1] >= 9 and "phys_sim_iteration" in pca_scores_df.columns:
        plt.figure(figsize=(9, 4))
        sns.lineplot(
            data=pca_scores_df.sort_values(["iteration", "beam_sub_iteration", "phys_sim_iteration"], na_position="last"),
            x="phys_sim_iteration",
            y="PC1",
            hue="iteration",
            style="beam_sub_iteration",
            marker="o",
        )
        plt.title("PC1 Trajectory Across Phys-Sim Iterations")
        plt.tight_layout()


In [ ]:
# Enrich PCA loadings with BeamNetworkFinal link attributes
from pilates.database.schema.beam_schema import BeamNetworkFinal
from sqlmodel import Session, text

if "pca_link_importance_df" not in globals() or pca_link_importance_df.empty:
    print("No PCA link-importance table found. Run the PCA exploratory cell first.")
else:
    network_view_name = f"v_beam_network_final_y{TARGET_YEAR}"
    tracker.create_grouped_view(
        view_name=network_view_name,
        schema=BeamNetworkFinal,
        namespace="beam",
        attach_facets=["artifact_family", "year", "iteration"],
        include_system_columns=True,
        mode="hybrid",
        if_exists="replace",
        missing_files="warn",
    )

    def q_ident(name: str) -> str:
        return '"' + str(name).replace('"', '""') + '"'

    with Session(tracker.engine) as session:
        info_rows = session.exec(text(f"PRAGMA table_info('{network_view_name}')")).all()
    colnames = [str(row[1]) for row in info_rows]

    def pick(*candidates):
        for c in candidates:
            if c in colnames:
                return c
        return None

    link_col = pick("link_id", "linkId")
    type_col = pick("attribute_orig_type", "attributeOrigType")
    lanes_col = pick("number_of_lanes", "numberOfLanes")
    cap_col = pick("link_capacity", "linkCapacity")

    if link_col is None:
        raise ValueError(f"Could not find link id column in {network_view_name}; columns={colnames}")

    select_parts = [f"CAST({q_ident(link_col)} AS BIGINT) AS link_id"]
    if type_col is not None:
        select_parts.append(f"ANY_VALUE({q_ident(type_col)}) AS attribute_orig_type")
    else:
        select_parts.append("NULL AS attribute_orig_type")
    if lanes_col is not None:
        select_parts.append(f"AVG(CAST({q_ident(lanes_col)} AS DOUBLE)) AS number_of_lanes")
    else:
        select_parts.append("NULL AS number_of_lanes")
    if cap_col is not None:
        select_parts.append(f"AVG(CAST({q_ident(cap_col)} AS DOUBLE)) AS link_capacity")
    else:
        select_parts.append("NULL AS link_capacity")

    sql = f"""
        SELECT
            {", ".join(select_parts)}
        FROM {q_ident(network_view_name)}
        WHERE {q_ident(link_col)} IS NOT NULL
        GROUP BY 1
    """
    with Session(tracker.engine) as session:
        network_rows = session.exec(text(sql)).all()

    network_link_df = pd.DataFrame(
        network_rows,
        columns=["link_id", "attribute_orig_type", "number_of_lanes", "link_capacity"],
    )
    network_link_df["link_id"] = pd.to_numeric(network_link_df["link_id"], errors="coerce").astype("Int64")

    pca_link_enriched_df = pca_link_importance_df.copy()
    pca_link_enriched_df["link_id"] = pd.to_numeric(pca_link_enriched_df["link"], errors="coerce").astype("Int64")
    pca_link_enriched_df = pca_link_enriched_df.merge(network_link_df, on="link_id", how="left")

    type_summary_df = (
        pca_link_enriched_df
        .groupby("attribute_orig_type", dropna=False)
        .agg(
            n_links=("link_id", "nunique"),
            mean_daily_volume=("daily_volume", "mean"),
            pc1_abs_loading_mean=("PC1_abs_loading_mean", "mean"),
            pc1_abs_loading_max=("PC1_abs_loading_max", "max"),
            mean_lanes=("number_of_lanes", "mean"),
            mean_capacity=("link_capacity", "mean"),
        )
        .reset_index()
        .sort_values("pc1_abs_loading_mean", ascending=False)
    )

    print("PCA link importance joined with network attributes:")
    display(pca_link_enriched_df.head(20))
    print("Functional-class summary (attribute_orig_type):")
    display(type_summary_df.head(20))

    top_types = type_summary_df.head(12).copy()
    if not top_types.empty:
        plt.figure(figsize=(10, 4))
        sns.barplot(
            data=top_types,
            x="attribute_orig_type",
            y="pc1_abs_loading_mean",
            color="#4C72B0",
        )
        plt.xticks(rotation=45, ha="right")
        plt.title("Mean |PC1 Loading| by attribute_orig_type")
        plt.tight_layout()


In [ ]:
# Consecutive phys-sim travel-time deltas within each (year, iteration, beam_sub_iteration) sequence
# Set USE_HOURLY_WEIGHTED = True to run the faster, hourly volume-weighted variant.
USE_HOURLY_WEIGHTED = True

if USE_HOURLY_WEIGHTED:
    delta_df = summarize_linkstats_traveltime_deltas_hourly_weighted(
        summary_df,
        tracker=tracker,
        exclude_zero_volume=True,
    )
    print(f"Computed {len(delta_df)} hourly-weighted travel-time delta rows")
else:
    delta_df = summarize_linkstats_traveltime_deltas(summary_df, tracker=tracker)
    print(f"Computed {len(delta_df)} travel-time delta rows")

delta_df


In [ ]:
# Plots
if delta_df.empty:
    print("delta_df is empty; skipping delta plots.")
else:
    sns.relplot(
        data=delta_df,
        x="phys_sim_iteration_prev",
        y="traveltime_delta_abs_mean",
        col="iteration",
        hue="beam_sub_iteration",
        kind="line",
        marker="o",
        facet_kws={"sharey": False},
    )

if "rollup" in globals() and not rollup.empty:
    sns.relplot(
        data=rollup,
        x="phys_sim_iteration",
        y="traveltime_mean_mean",
        col="iteration",
        hue="beam_sub_iteration",
        kind="line",
        marker="o",
    )

    sns.relplot(
        data=rollup,
        x="phys_sim_iteration",
        y="total_delay_hours_total",
        col="iteration",
        hue="beam_sub_iteration",
        kind="line",
        marker="o",
    )

    sns.relplot(
        data=rollup,
        x="phys_sim_iteration",
        y="vmt_per_vht_mph_total",
        col="iteration",
        hue="beam_sub_iteration",
        kind="line",
        marker="o",
    )

    sns.relplot(
        data=rollup,
        x="phys_sim_iteration",
        y="vmt_miles_total",
        col="iteration",
        hue="beam_sub_iteration",
        kind="line",
        marker="o",
        facet_kws={"sharey": False},
    )

    plt.figure(figsize=(8, 5))
    sns.scatterplot(
        data=rollup,
        x="vmt_miles_total",
        y="traveltime_mean_mean",
        hue="beam_sub_iteration",
        style="iteration",
        s=70,
    )
    plt.title("Travel Time vs VMT by Iteration/Sub-Iteration")
    plt.tight_layout()

if not delta_df.empty and {"prev_volume_total", "curr_volume_total"}.issubset(delta_df.columns):
    melt_df = delta_df.melt(
        id_vars=["iteration", "beam_sub_iteration", "phys_sim_iteration_prev"],
        value_vars=["prev_volume_total", "curr_volume_total"],
        var_name="period",
        value_name="volume_total",
    )
    sns.relplot(
        data=melt_df,
        x="phys_sim_iteration_prev",
        y="volume_total",
        hue="period",
        col="beam_sub_iteration",
        row="iteration",
        kind="line",
        marker="o",
        facet_kws={"sharey": False},
    )


## Notes

- This notebook intentionally mounts `workspace://` to the archived run directory.
- It computes metrics from a single Consist grouped hybrid view (`tracker.create_grouped_view(...)`) rather than direct parquet paths.
- If view creation fails, verify mounts and ensure your Consist DB schema matches your Consist package version.
- You can switch artifact family to `linkstats_parquet` or `linkstats` in `find_linkstats_artifacts(...)` for other analyses.
